# Word Embeddings with `Word2Vec`

So far, you have performed machine learning with documents. Similarities of words or multi-word combinations have not played a role.

Now you are trying to find a representation of words that allows you to compute similarities between them.

## Load data

As usual, you load the linguistically analyzed data. This time, we going to use a corpus of 85 Wikipedia articles on software security.

In [1]:
#!pip install "gensim>=4.0.0"

In [2]:
import sqlite3 
import pandas as pd

sql = sqlite3.connect("wiki_articles_swsec_extended.db")

df = pd.read_sql_query("SELECT * from wiki_articles_swsec_extended", sql)
df

,title,text,name,url,datePublished,dateModified,headline,nouns,adjectives,verbs,lemmas,nav,entities,noun_chunks,no_tokens,no_sentences,no_noun_chunks
0,Abuse case,"From Wikipedia, the free encyclopedia\n\n\nAbu...",Abuse case,https://en.wikipedia.org/wiki/Abuse_case,2010-03-19T13:30:06Z,2021-10-30T20:20:23Z,specification model for security requirements ...,abuse|case|specification|model|security|requir...,complete|more|harmful#just|coherent#instead|ac...,be|use#be#be|introduce|work#define|be|be#can|d...,|abuse|case|be|a|specification|model|for|secu...,abuse|case|be|specification|model|security|req...,John McDermott|Chris Fox|1999|Computer Science...,Abuse case|a specification model|security req...,351.0,17.0,71.0
1,Access-control list,"From Wikipedia, the free encyclopedia\n\n\nLis...",Access-control list,https://en.wikipedia.org/wiki/Access-control_list,2002-07-12T08:06:50Z,2024-08-11T10:37:11Z,list associated with a computing system resour...,list|permission|system|resource|computer|secur...,as|well#typical#only#many|historical|first#usu...,be|associate#specify|be|grant|be|allow|give#sp...,|list|of|permission|for|a|system|resource|in|...,list|permission|system|resource|computer|secur...,ACL#Alice|Bob|Alice|Bob#CLASS(TSOAUTH#ACLs|fir...,List|permissions|a system resource|computer s...,1163.0,51.0,247.0
2,Antivirus software,"From Wikipedia, the free encyclopedia\n\n\nCom...",Antivirus software,https://en.wikipedia.org/wiki/Antivirus_software,2003-07-16T00:29:07Z,2024-09-05T15:59:54Z,computer security software that is used to pre...,computer|software|computer|virus|antivirus#med...,malicious|here#antiviral#open|originally#also#...,defend|redirect#see#base|be|develop#abbreviate...,|computer|software|to|defend|against|maliciou...,computer|software|defend|malicious|computer|vi...,ClamTk|ClamAV|Tomasz Kojm|2001#1971–1980|pre-a...,Computer software|malicious computer viruses|...,6168.0,258.0,1211.0
3,Application security,"From Wikipedia, the free encyclopedia\n\n\nMea...",Application security,https://en.wikipedia.org/wiki/Application_secu...,2005-08-29T21:54:46Z,2024-09-04T21:24:46Z,measures taken to improve the security of an a...,measure|security|application|application|secur...,short|secure#final|preferably#whole|as|well#sp...,take|improve|include|introduce#be|improve|find...,|measure|take|to|improve|the|security|of|an|a...,measure|take|improve|security|application|appl...,AppSec#Android Applications Web Application Se...,Measures|the security|an application Applicat...,963.0,47.0,193.0
4,Application firewall,"From Wikipedia, the free encyclopedia\n\n\nLay...",Application firewall,https://en.wikipedia.org/wiki/Application_fire...,2004-09-04T01:27:44Z,2024-04-29T22:37:28Z,a form of firewall that controls input/output ...,Layer|application|layer|network|security|syste...,primary#additional#reliable#unsourced#generall...,be#see#need#help|improve|add#may|be|challenge|...,|Layer|7|/|application|layer|network|security...,Layer|application|layer|network|security|syste...,Firewall#Unsourced#JSTOR|February 2010#two#Gen...,Layer 7/application layer network security sy...,1115.0,46.0,199.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,Threat model,"From Wikipedia, the free encyclopedia\n\n\nPro...",Threat model,https://en.wikipedia.org/wiki/Threat_model,2006-04-04T11:57:04Z,2024-09-05T19:37:03Z,"process by which potential threats, such as st...",process|vulnerability|threat|modeling|process|...,structural|potential|such|structural|appropria...,identify|be|can|be|identify|enumerate|prioriti...,|process|of|identify|structural|vulnerability...,process|identify|structural|vulnerability|thre...,morning#the early 1960s#first|Christopher Alex...,Process|structural vulnerabilities Threat mod...,2088.0,99.0,406.0
81,Trojan horse (computing),"From Wikipedia, the free encyclopedia\n\n\nTyp...",Trojan horse (computing),https://en.wikipedia.org/wiki/Trojan_horse_(co...,2001-09-28T02:09:56Z,2024-09-08T09:42:07Z,"A Trojan hors

## Train Embeddings

Word embeddings are best calculated with `gensim`. As usual you have to transform the documents into a double nested array.

In [3]:
import regex as re
from spacy.lang.en.stop_words import STOP_WORDS as stop_words
gensim_words = [[w for w in re.split(r'[\\|\\#]', doc.lower()) if w not in stop_words]
                    for doc in df["nav"]]

Then you can calculate the embedding. Here it is set so that only words are considered that occur at least five times: 

In [4]:
from gensim.models import Word2Vec
from gensim.models import KeyedVectors

#w2v = Word2Vec(gensim_words, min_count=5)
#w2v.wv.save_word2vec_format("wiki_articles_swsec_extended.w2v")

In [5]:
w2v = KeyedVectors.load_word2vec_format("wiki_articles_swsec_extended.w2v")

## Query similarities

Now you can start with simple similarity queries. If you are interested in what is similar to `software`, you can try the following query:

In [6]:
w2v.most_similar(positive=["software"], topn=10)

[('product', 0.987931489944458),
 ('agile', 0.9864972233772278),
 ('cycle', 0.9859892129898071),
 ('dynamic', 0.9852984547615051),
 ('static', 0.9852288961410522),
 ('design', 0.9848576188087463),
 ('test', 0.9846635460853577),
 ('defect', 0.9846277236938477),
 ('life', 0.9845686554908752),
 ('quality', 0.9844837784767151)]

The result is reasonable.

Try it again with `python`:

In [7]:
w2v.most_similar(positive=["python"], topn=10)

[('likelihood', 0.9948380589485168),
 ('military', 0.9946828484535217),
 ('c', 0.9946693778038025),
 ('analyst', 0.9946663975715637),
 ('point', 0.9946562051773071),
 ('characteristic', 0.9945819973945618),
 ('stakeholder', 0.9945654273033142),
 ('impossible', 0.9945646524429321),
 ('expert', 0.9945233464241028),
 ('agent', 0.9945124983787537)]

Word2Vec, like all embeddings, can also work with addition and subtraction of vectors and draw analogy conclusions from them.

Consider the following equation:

software - security = ? - release

By rearranging you can solve for ?

? = software + release - security

What is the result of `Word2Vec`?

In [8]:
pd.DataFrame(w2v.most_similar(positive=["software", "release"], 
                                   negative=["security"],  topn=10))

,0,1
0,malware,0.976546
1,spyware,0.973206
2,virus,0.968284
3,program,0.965959
4,cross,0.965095
5,antivirus,0.963895
6,non,0.959177
7,keylogger,0.958271
8,scripting,0.956749
9,site,0.953785


That is a nice result: Software without security will incur problems at the time of release.

If you have forgotten the name of Google's mobile operating system, you can ask `Word2Vec`:

In [9]:
pd.DataFrame(w2v.most_similar(positive=["ios", "google"], 
                                   negative=["apple"],  topn=10))

,0,1
0,come,0.997973
1,delete,0.997940
2,android,0.997820
3,text,0.997812
4,money,0.997806
5,drive,0.997798
6,ransom,0.997787
7,contain,0.997781
8,certain,0.997774
9,connection,0.997771


The result is OK, too. And this was all trained without supervision!

## Phrases

`gensim` can identify phrases in the corpus and use them instead of tokens as vocabulary for training:

In [10]:
from gensim.models import Phrases
entity_transformer = Phrases(gensim_words)
w2vp = Word2Vec(entity_transformer[gensim_words], min_count=5)

Here is the result for `software`:

In [11]:
w2vp.wv.most_similar(positive=["software"], topn=15)

[('bug', 0.9987013339996338),
 ('hardware', 0.9984827041625977),
 ('design', 0.9984740018844604),
 ('protection', 0.9983307123184204),
 ('release', 0.9982061982154846),
 ('component', 0.9981834888458252),
 ('antivirus_software', 0.9981800317764282),
 ('product', 0.9981521368026733),
 ('application', 0.998133659362793),
 ('behavior', 0.9981324076652527),
 ('update', 0.9981114268302917),
 ('exist', 0.9980964064598083),
 ('common', 0.9980883002281189),
 ('functionality', 0.9980804324150085),
 ('purpose', 0.9980778694152832)]

You can now see that phrases are included and marked by `_`. It is hard to decide if the result is really better. For 'python', it is clearly not the case.

In [12]:
w2vp.wv.most_similar(positive=["python"], topn=10)

[('support', 0.9967483282089233),
 ('sale', 0.9966962337493896),
 ('term', 0.9966620206832886),
 ('classification', 0.996620237827301),
 ('study', 0.9966155886650085),
 ('approach', 0.9966011643409729),
 ('propose', 0.9965834617614746),
 ('completely', 0.9965634346008301),
 ('build', 0.9965556263923645),
 ('good_practice', 0.9965289831161499)]

## *Word2Vec* as the most common application of Word Embeddings

Often, *Word2Vec* is used almost synonymously with Word Embeddings - admittedly it was also the first Word Embedding, which brought it to a significant spread. The performance is quite good, both the calculation and the query can be done in (almost) real time (the query is equally fast for all embeddings).

Depending on the use case, it is a good idea to start with Word2Vec and use that as a baseline. If the other embeddings you will get to know later do not bring you any advantages, just stay with Word2Vec.